In [ ]:
import os
import pandas as pd

# 1) READ MULTIPLE CSV FILES AND EXTRACT PZ, PX
downloads_folder = r"C:\Users\TrevorWhite\Downloads"
file_names = [
    "MW_PBP_24.csv",
    "WCC_PBP_24.csv",
    "SEC_PBP_24.csv",
    "P12_PBP_24.csv",
    "B10_PBP_24.csv",
    "BE_PBP_24.csv",
    "B12_PBP_24.csv",
    "ACC_PBP_24.csv"
]

data_list = []
for file_name in file_names:
    file_path = os.path.join(downloads_folder, file_name)
    if os.path.exists(file_path):
        df_temp = pd.read_csv(file_path)
        data_list.append(df_temp)
        print(f"Successfully read: {file_name}")
    else:
        print(f"File not found: {file_name}")

combined_df = pd.concat(data_list, ignore_index=True)
print("All files successfully appended into a single DataFrame.")
print("Combined DataFrame shape:", combined_df.shape)

# List of columns to select (only uniqPitchId, PX, PZ)
pz_px_columns = ["uniqPitchId", "PZ", "PX", "batterHand"]
selected_columns_df = combined_df[pz_px_columns].copy()
print(selected_columns_df.head())

# 2) LOAD TRAINING DATA THAT NEEDS PX, PZ MERGED IN
train_data_path = r"C:\Users\TrevorWhite\Downloads\NCAA_STUFF_PLUS_24_TRAIN.csv"
train_df = pd.read_csv(train_data_path)
print("Train Data shape:", train_df.shape)
print(train_df.head())

# 3) MERGE PX, PZ INTO THE TRAIN DATA ON 'uniqPitchId'
# If 'uniqPitchId' is unique in selected_columns_df, it will attach PZ, PX accordingly.
df = pd.merge(
    train_df,
    selected_columns_df,
    on='uniqPitchId',
    how='left'   # or 'inner' if you only want matching rows
)

# print("Merged DataFrame shape:", df.shape)
train_df.head()

# You can now work with 'df' which contains all columns from train_df plus PZ and PX.


In [ ]:
# Multiply PX by -1 for all rows because trackman and trumedia are opposite
df = train_df
df['PX'] = df['PX'] * -1


#NOW WE KNOW THAT A NEGATIVE PX IS INSIDE ON A LEFTY

In [ ]:
import numpy as np
import pandas as pd

def add_mph_adjustments(df, 
                        velocity_col='start_speed', 
                        px_col='PX', 
                        pz_col='PZ',
                        stand_col='batterHand'):
    """
    Adds columns 'mph_adjustment' and 'EffectiveVelocity' to the DataFrame df
    based on the (px, pz) location of each pitch and the batter's handedness.
    
    - The 5×5 'mph_adjustments' array is defined for a left-handed batter 
      (from catcher's perspective).
    - If batter is right-handed, we flip px by negating it so that 'inside' 
      remains the left side of the array.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame of pitch data that must contain:
          - A velocity_col (default 'start_speed') for velocity (mph)
          - px_col (default 'PX') for horizontal location
          - pz_col (default 'PZ') for vertical location
          - stand_col (default 'batterHand') in {'L', 'R'}
    velocity_col : str
        Column name for pitch velocity (mph).
    px_col : str
        Column name for horizontal location of the pitch.
    pz_col : str
        Column name for vertical location of the pitch.
    stand_col : str
        Column name for batter's handedness: 'L' or 'R'.

    Returns
    -------
    pd.DataFrame
        The input DataFrame with two new columns: 'mph_adjustment' and
        'EffectiveVelocity'.
    """
    
    # 5×5 mph-adjustment grid (for a LHB view: row=0 => TOP, col=0 => LEFT).
    mph_adjustments = np.array([
        [ 0.84,  0.68,  0.52,  0.34,  0.15],  # row=0 (high in zone)
        [ 0.72,  0.56,  0.39,  0.21,  0.03],
        [ 0.46,  0.29,  0.13, -0.04, -0.22],
        [ 0.31,  0.15, -0.02, -0.18, -0.36],
        [ 0.22,  0.07, -0.09, -0.26, -0.44]   # row=4 (low in zone)
    ])

    # Basic strike-zone dimensions:
    x_min, x_max = -1.5,  1.5  # left-right
    y_min, y_max =  0.75, 4.0  # bottom-top
    
    # Extended/clamped boundaries
    ext_x_min, ext_x_max = -1.67, 1.67
    ext_y_min, ext_y_max =  0.50, 4.33

    # 5×5 means n_rows=5, n_cols=5
    n_rows, n_cols = mph_adjustments.shape
    
    # Size of each cell in X and Z
    cell_width  = (x_max - x_min) / n_cols   # e.g. 3.0 / 5 = 0.6
    cell_height = (y_max - y_min) / n_rows   # e.g. 3.25 / 5 = 0.65

    def get_mph_adjustment(px_val, pz_val, stand):
        # Flip horizontal axis if batter is right-handed
        if stand == 'R':
            px_val = -px_val

        # 1) Clamp px, pz within extended bounds
        px_c = np.clip(px_val, ext_x_min, ext_x_max)
        pz_c = np.clip(pz_val, ext_y_min, ext_y_max)

        # 2) Clamp again for grid indexing (the main zone)
        px_base = np.clip(px_c, x_min, x_max)
        pz_base = np.clip(pz_c, y_min, y_max)

        # 3) Convert px_base -> col index in [0..4]
        col_float = (px_base - x_min) / cell_width
        col_idx   = int(np.floor(col_float))
        col_idx   = max(0, min(col_idx, n_cols - 1))

        # 4) Convert pz_base -> row index in [0..4], top => row=0
        #    so near pz=4 => row_float ~ 0 => row_idx=0
        row_float = (y_max - pz_base) / cell_height
        row_idx   = int(np.floor(row_float))
        row_idx   = max(0, min(row_idx, n_rows - 1))

        # Return the actual mph adjustment
        return mph_adjustments[row_idx, col_idx]

    # Compute mph_adjustment for each pitch
    df['mph_adjustment'] = df.apply(
        lambda row: get_mph_adjustment(
            row[px_col], row[pz_col], row[stand_col]
        ),
        axis=1
    )

    # Compute EffectiveVelocity = velocity + mph_adjustment
    df['EffectiveVelocity'] = df[velocity_col] + df['mph_adjustment']

    return df


In [ ]:
df[df["pitch_type"] == "SL"].head(111)


In [ ]:
df = df.dropna(subset=['PX', 'PZ', 'batterHand', 'start_speed'])
# 1) Add mph_adjustment + EffectiveVelocity columns
df = add_mph_adjustments(
    df, 
    velocity_col='start_speed', 
    px_col='PX', 
    pz_col='PZ'
)
df.head()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm

def plot_mph_adjustment_scatter(df, px_col='PX', pz_col='PZ'):
    """
    Plots the pitches' px vs. pz, colored by the mph_adjustment column.
    """
    fig, ax = plt.subplots(figsize=(8, 8))

    # Normalize color scale around the min/max of mph_adjustment
    norm = Normalize(vmin=df['mph_adjustment'].min(), 
                     vmax=df['mph_adjustment'].max())
    cmap = cm.get_cmap('coolwarm')

    sc = ax.scatter(df[px_col], df[pz_col],
                    c=df['mph_adjustment'], 
                    cmap=cmap, 
                    norm=norm, 
                    edgecolor='black', 
                    alpha=0.8)

    # Colorbar
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label('MPH Adjustment')

    # Plot some reference lines for the zone
    ax.add_patch(plt.Rectangle((-0.83, 1.5), 
                               1.66, 2.1, 
                               fill=False, 
                               edgecolor='black', 
                               linewidth=2))
    ax.set_xlim([-1.67, 1.67])
    ax.set_ylim([0.5, 4.33])
    ax.set_xlabel("Horizontal Location (px)")
    ax.set_ylabel("Vertical Location (pz)")
    ax.set_title("MPH Adjustment by Pitch Location")
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(0, 5)

    plt.show()

# Example usage:
# 1) df_pitches = add_mph_adjustments(df_pitches)
# 2) plot_mph_adjustment_scatter(df_pitches)


In [ ]:
df_left.sort_values(by="mph_adjustment", ascending=False).head()


In [ ]:
# Split data into left and right handed batters
df_left = df[df['batterHand'] == 'L']
df_right = df[df['batterHand'] == 'R']



# 2) Plot the mph_adjustment results
plot_mph_adjustment_scatter(
    df_right, 
    px_col='PX', 
    pz_col='PZ'
)


In [ ]:
plot_mph_adjustment_scatter(
    df_right, 
    px_col='PX', 
    pz_col='PZ'
)

In [ ]:
import numpy as np

# # Add these pandas display options at the beginning of your notebook, after the imports
# pd.set_option('display.max_rows', None)  # Show all rows
# pd.set_option('display.max_columns', None)  # Show all columns
# pd.set_option('display.width', None)  # Width of the display in characters
# pd.set_option('display.max_colwidth', None)  # Show full content of each column

# Create the 'same_side' column
df['same_side'] = np.where(df['batterHand'] == df['pitcher_hand'], 1, 0)




# Multiply PX by -1 where pitcher_hand is 'L'
df['PX'] = np.where(df['pitcher_hand'] == 'L', df['PX'] * -1, df['PX'])
df['HorzApprAngle'] = np.where(df['pitcher_hand'] == 'L', df['HorzApprAngle'] * -1, df['HorzApprAngle'])

# ... existing code ...
df.head(111)

###+PX: inside on righty
    #-px:inside on lefty





In [ ]:
#need two plate cols - one to represent proximity to batter - flipped on batter  -----if applied it needs same_side seperation
#one to represent throwing to side of handedness  -----if applied it needs vs RHH or LHH seperation
df['PX_B'] = np.where(df['pitcher_hand'] == 'L', df['PX'] * -1, df['PX'])
df['PX_B'] = np.where(df['batterHand'] == 'R', df['PX_B'] * -1, df['PX_B'])#ALIGN ON BATTER
df['ax_B'] = np.where(df['pitcher_hand'] == 'L', df['ax'] * -1, df['ax'])
df['ax_B'] = np.where(df['batterHand'] == 'R', df['ax_B'] * -1, df['ax_B'])
df['HorzApprAngle_B'] = np.where(df['pitcher_hand'] == 'L', df['HorzApprAngle'] * -1, df['HorzApprAngle'])
df['HorzApprAngle_B'] = np.where(df['batterHand'] == 'R', df['HorzApprAngle'] * -1, df['HorzApprAngle'])

##ALL + X MOVEMENT ARE TOWARDS LHB WHILE - TOWARDS RHB

#PX_B now represents how close(-) or far(+) the ball is from the hitter SO IF THE DIFF IS POSITIVE THAT MEANS IT IS GETTTING FURTHER AWAY ELSE CLOSER


In [ ]:
import pandas as pd
import numpy as np

def add_pitch_differences(df):
    """
    Given a DataFrame with `uniqPitchid` as "Gameid-Plateappearanceno-pitchno",
    add:
      - is_first_pitch (1 if first pitch in a plate appearance, else 0)
      - Differences from previous pitch for the columns:
          ["start_speed", "EffectiveVelo", "az", "ax", "x0", "z0", "PX", "PZ"]

    Returns the modified DataFrame.
    """
    # 1) Split 'uniqPitchid' into separate columns: Gameid, PlateApp, PitchNo
    #    Example: "20231208-001-3" => Gameid="20231208", PlateApp="001", PitchNo="3"
    df[['Gameid', 'PlateApp', 'PitchNo']] = df['uniqPitchId'].str.split('-', expand=True)
    
    # Convert PitchNo to integer for proper sorting
    df['PitchNo'] = df['PitchNo'].astype(int)
    
    # 2) Sort so we can do diffs in correct order
    df.sort_values(by=['Gameid', 'PlateApp', 'PitchNo'], inplace=True)
    
    # 3) Create `is_first_pitch` indicator
    #    It's the first pitch if PitchNo == 1
    df['is_first_pitch'] = np.where(df['PitchNo'] == 1, 1, 0)
    
    # 4) For each numeric column, create a diff from the prior pitch in the SAME game & plate appearance
    cols_to_diff = ["start_speed", "EffectiveVelocity", "az", "ax_B","PX_B" ,"PZ", "HorzApprAngle_B","VertApprAngle"]

    # Convert all columns in cols_to_diff to numeric (handle any strings or missing values)
    for col in cols_to_diff:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Group by game + plate appearance, then do diff
    grouping = df.groupby(['Gameid', 'PlateApp'])
    
    for col in cols_to_diff:
        diff_col = f"{col}_diff_LP"
        df[diff_col] = grouping[col].diff()
        # The first pitch in each group will be NaN for the diff
    
    return df

# USAGE EXAMPLE:
df = add_pitch_differences(df)
df.head()


In [ ]:
#AX_B - NEGATIVE= AWAY FROM BATTER POSITIVE = TOWARDS
#HAA_B - NEGATIVE = AWAY FROM BATTER POSITIVE = TOWARDS
#PX - #PX_B now represents how close(-) or far(+) the ball is from the hitter SO IF THE DIFF IS POSITIVE THAT MEANS IT IS GETTTING FURTHER AWAY ELSE CLOSER

In [ ]:
#######disclude waste pitches
#####evaluate positive diffs and negative diffs seperatley
#####plot corr vs target

I want to see this displayed as when var increases vs when var decreases the corr is and the p value is,, maybe what i really want is a linear regression for each case  -----

start_speed_diff_LP
	EffectiveVelocity_diff_LP	
    az_diff_LP	
    ax_diff_B_LP(same_side)		##want to know effect of moving pitch closer vs further away from batter
                                #should ax actually be flipped in respect to batter than pitcher when predicting ideal pitch location?
                                #currently we deal with pitchers movement by arm-side vs glove -- with same_side var we capture whether this arm-side is towards or away from batter
PX_B_diff_LP(controlling for same_side)

PZ_diff_LP 
HorzApprAngle_diff_LP		
VertApprAngle_B_diff_LP



In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

def analyze_diffs_with_sameside_in_regression(df, 
                                              target_col='target', 
                                              same_side_col='same_side'):
    """
    1) Exclude pitches outside CHASE_SIDE=(-1.67,1.67), CHASE_HEIGHT=(0.5,4.33).
    2) For each diff variable, split into positive vs. negative.
    3) Compute simple Pearson correlation of (var, target).
    4) Fit a linear regression:
       - If var is one of the "B" columns, include same_side as an additional regressor.
       - Otherwise, use only var as the regressor.

    Returns a DataFrame with correlation + regression results.
    """

    # 1) Filter out "waste pitches"
    side_min, side_max = -1.67, 1.67
    height_min, height_max = 0.5, 4.33

    df_filtered = df[
        (df['PX'].between(side_min, side_max)) &
        (df['PZ'].between(height_min, height_max))
    ].copy()

    # 2) The columns to analyze
    vars_to_check = [
        "start_speed_diff_LP",
        "EffectiveVelocity_diff_LP",
        "az_diff_LP",
        "ax_B_diff_LP",         # B-type => use same_side in regression
        "PX_B_diff_LP",         # B-type => use same_side
        "PZ_diff_LP",
        "HorzApprAngle_B_diff_LP",
        "VertApprAngle_diff_LP"  # B-type => use same_side
    ]

    # These are the variables that require adding `same_side` as a second predictor
    vars_need_sameside = {
        "ax_B_diff_LP",
        "PX_B_diff_LP",
        "HorzApprAngle_B_diff_LP"
        # If you also want "VertApprAngle_diff_LP" in the B set, add it here
    }

    results = []

    for var in vars_to_check:
        # Split data into positive and negative for this var
        df_pos = df_filtered[df_filtered[var] > 0]
        df_neg = df_filtered[df_filtered[var] < 0]

        def analyze_subset(subset, direction):
            """Compute correlation (var vs. target), then fit regression:
               - single regressor (var)
               - multiple (var + same_side) if needed.
               Drop rows with inf/NaN before stats."""
            
            # --- NEW CODE to drop NaNs/∞ in needed columns ---
            columns_needed = [var, target_col]
            if var in vars_need_sameside:
                columns_needed.append(same_side_col)
                
            # Replace ±∞ with NaN, then drop all NaN
            subset_clean = subset[columns_needed].replace([np.inf, -np.inf], np.nan).dropna()
            # If there's not enough data left, bail out
            if len(subset_clean) < 2:
                return {
                    'Variable': var,
                    'Direction': direction,
                    'Count': len(subset_clean),
                    'Correlation': np.nan,
                    'CorrPval': np.nan,
                    'Slope_var': np.nan,
                    'Slope_var_pval': np.nan,
                    'Slope_same_side': np.nan,
                    'Slope_same_side_pval': np.nan,
                    'Intercept': np.nan,
                    'R2': np.nan
                }

            # -- Pearson correlation (raw) --
            corr, pval = stats.pearsonr(subset_clean[var], subset_clean[target_col])

            # -- Regression --
            if var in vars_need_sameside:
                # Multiple regression: target ~ var + same_side
                X = subset_clean[[var, same_side_col]].copy()
                # Make sure same_side is numeric (0/1)
                if X[same_side_col].dtype == bool:
                    X[same_side_col] = X[same_side_col].astype(int)
            else:
                # Single regressor
                X = subset_clean[[var]]

            # Add intercept
            X = sm.add_constant(X)
            y = subset_clean[target_col]

            model = sm.OLS(y, X, missing='drop').fit()

            # Extract params
            intercept = model.params.get('const', np.nan)
            slope_var = model.params.get(var, np.nan)
            slope_var_pval = model.pvalues.get(var, np.nan)

            # If same_side is in the model:
            slope_same_side = (
                model.params.get(same_side_col, np.nan)
                if var in vars_need_sameside else np.nan
            )
            slope_same_side_pval = (
                model.pvalues.get(same_side_col, np.nan)
                if var in vars_need_sameside else np.nan
            )

            return {
                'Variable': var,
                'Direction': direction,
                'Count': len(subset_clean),
                'Correlation': corr,
                'CorrPval': pval,
                'Slope_var': slope_var,
                'Slope_var_pval': slope_var_pval,
                'Slope_same_side': slope_same_side,
                'Slope_same_side_pval': slope_same_side_pval,
                'Intercept': intercept,
                'R2': model.rsquared
            }

        # Analyze positive subset
        result_pos = analyze_subset(df_pos, 'positive')
        results.append(result_pos)

        # Analyze negative subset
        result_neg = analyze_subset(df_neg, 'negative')
        results.append(result_neg)

    return pd.DataFrame(results)

# ----------------------
#USAGE EXAMPLE
df_results = analyze_diffs_with_sameside_in_regression(
    df,
    target_col='target',
    same_side_col='same_side'
)
print(df_results)


In [ ]:
df_results.head(111)